# 26 · Threshold Tuning & Calibration

**2026 learning module** · Run cells from top to bottom. This notebook uses built-in/sample datasets and fixed random seeds where appropriate.

## Goal
See how changing a classification threshold changes precision and recall. The threshold must be selected using validation data, not the final test set.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, PrecisionRecallDisplay, brier_score_loss
from sklearn.calibration import CalibrationDisplay

X, y = make_classification(n_samples=3500, n_features=18, n_informative=7, n_redundant=4, weights=[0.88, 0.12], class_sep=0.9, random_state=2026)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=2026)
X_fit, X_valid, y_fit, y_valid = train_test_split(X_train, y_train, test_size=0.25, stratify=y_train, random_state=12)
model = HistGradientBoostingClassifier(max_iter=180, learning_rate=0.08, random_state=2026)
model.fit(X_fit, y_fit)
valid_proba = model.predict_proba(X_valid)[:, 1]

In [ ]:
rows = []
for threshold in np.arange(0.1, 0.91, 0.05):
    p = (valid_proba >= threshold).astype(int)
    rows.append({"threshold": threshold, "precision": precision_score(y_valid, p, zero_division=0), "recall": recall_score(y_valid, p, zero_division=0), "f1": f1_score(y_valid, p, zero_division=0)})
thresholds = pd.DataFrame(rows)
thresholds.set_index("threshold").plot(figsize=(9, 5), marker="o", title="Validation metrics by threshold");
plt.grid(alpha=0.25); plt.tight_layout()
best_threshold = float(thresholds.loc[thresholds["f1"].idxmax(), "threshold"])
print("Chosen threshold from validation F1:", round(best_threshold, 2))

In [ ]:
# Final test evaluation only after threshold selection.
model.fit(X_train, y_train)
test_proba = model.predict_proba(X_test)[:, 1]
test_pred = (test_proba >= best_threshold).astype(int)
print({"precision": round(precision_score(y_test, test_pred, zero_division=0), 3),
       "recall": round(recall_score(y_test, test_pred, zero_division=0), 3),
       "f1": round(f1_score(y_test, test_pred, zero_division=0), 3),
       "brier_score": round(brier_score_loss(y_test, test_proba), 3)})
CalibrationDisplay.from_predictions(y_test, test_proba, n_bins=10); plt.tight_layout()

## Key idea
The best threshold depends on the costs of false positives and false negatives. F1 is just one possible objective; choose a metric and threshold that match the real task. Calibration plots assess probability reliability, not classification accuracy.